In [2]:
import pandas as pd 

data_path = r"C:\Users\USER\Desktop\machine_learning\ai_fraud_detector_streamlit\ai-fraud-detector\backend\data\raw\email_dataset2.csv"

df = pd.read_csv(data_path)
df

C:\Users\USER\AppData\Local\Temp\ipykernel_14848\1148454610.py:5: DtypeWarning: Columns (2,3,4,5,6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path)


,label,text,URL,EMAIL,PHONE,lang,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10
0,ham,Your opinion about me? 1. Over 2. Jada 3. Kusr...,No,No,No,en,NaN,NaN,NaN,NaN,NaN
1,ham,What's up? Do you want me to come online? If y...,No,No,No,en,NaN,NaN,NaN,NaN,NaN
2,ham,So u workin overtime nigpun?,No,No,No,en,NaN,NaN,NaN,NaN,NaN
3,ham,"Also sir, i sent you an email about how to log...",No,No,No,en,NaN,NaN,NaN,NaN,NaN
4,spam,Please Stay At Home. To encourage the notion o...,No,No,No,en,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
138808,spam,Hey little one! Exciting news! Mama and baby a...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
138809,spam,Amazing DATA deals on your Pulse Plan today! D...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
138810,spam,Special offer just for you! Get 1GB @15 bob va...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
138811,spam,NEW ARRIVAL - JUNE 23RD Dresses @ 300; Kondel...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
def function_transform(x):
    if x == 'spam' or 'Spam':
        return 1
    elif x == 'ham':
        return 0
    return x

df = df[['text', 'label']]
# Call apply on the specific column, not the whole dataframe
df['label'] = df['label'].apply(function_transform)

df['label'] = df['label'].astype(int)

df

,text,label
0,Your opinion about me? 1. Over 2. Jada 3. Kusr...,1
1,What's up? Do you want me to come online? If y...,1
2,So u workin overtime nigpun?,1
3,"Also sir, i sent you an email about how to log...",1
4,Please Stay At Home. To encourage the notion o...,1
...,...,...
138808,Hey little one! Exciting news! Mama and baby a...,1
138809,Amazing DATA deals on your Pulse Plan today! D...,1
138810,Special offer just for you! Get 1GB @15 bob va...,1
138811,NEW ARRIVAL - JUNE 23RD Dresses @ 300; Kondel...,1


In [9]:
df_1 = df

df_1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138813 entries, 0 to 138812
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   text    138812 non-null  object
 1   label   138813 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 2.1+ MB


In [8]:
data_path_2 = r"..\data\raw\email_dataset.csv"

df_2 = pd.read_csv(data_path_2)

df_2 = df_2[['text', 'label']]

df_2

,text,label
0,Subject: Office maintenance\n\nThanks for your...,0
1,"Hello, your profile has been locked. Use the s...",1
2,"Hi there, congratulations! You are the winner ...",1
3,"Attention, this is the fraud prevention accoun...",1
4,"Notice, your profile has been restricted. Use ...",1
...,...,...
9995,Subject: Code review summary\n\nI booked the l...,0
9996,"Hello, we talked about meeting again after the...",1
9997,"Hi there, we talked about meeting again after ...",1
9998,"Dear user, this is an expires midnight notice ...",1


In [10]:
#stack df1 to df2

stacked_df = pd.concat([df_1,df_2], axis = 0)

len(stacked_df)

148813

In [27]:
def clean_text(text):
    """
    Advanced text cleaning for phishing detection:
    1. Remove URLs but keep domain info
    2. Normalize whitespace and special patterns
    3. Convert to lowercase
    4. Tokenize and remove stopwords
    5. Keep important phishing indicators (currency, numbers)
    """
    if not isinstance(text, str):
        return ""
    
    # Replace URLs with URL token to preserve URL presence signal
    text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', 'URL', text)
    
    # Replace email addresses with EMAIL token
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', 'EMAIL', text)
    
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove special characters but keep alphanumeric, space, and special tokens
    text = re.sub(r'[^a-z0-9\s$€£¥%()[\]\-]', ' ', text)
    
    # Normalize repeated characters (e.g., "!!!!" -> "!")
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stopwords but keep important phishing indicators
    stop_words = set(stopwords.words('english'))
    important_tokens = {'url', 'email', 'click', 'verify', 'confirm', 'urgent', 'act', 'now', 'expire'}
    
    tokens = [
        token for token in tokens 
        if (token not in stop_words or token in important_tokens) and len(token) > 1
    ]
    
    return " ".join(tokens)

In [28]:
cleaned_df = clean_text(stacked_df)

In [31]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# You must also download the NLTK resources if you haven't already
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [32]:
stacked_df['cleaned_text'] = stacked_df['text'].apply(clean_text)

In [33]:


stacked_df.dropna()



,text,label
0,Your opinion about me? 1. Over 2. Jada 3. Kusr...,1
1,What's up? Do you want me to come online? If y...,1
2,So u workin overtime nigpun?,1
3,"Also sir, i sent you an email about how to log...",1
4,Please Stay At Home. To encourage the notion o...,1
...,...,...
9995,Subject: Code review summary\n\nI booked the l...,0
9996,"Hello, we talked about meeting again after the...",1
9997,"Hi there, we talked about meeting again after ...",1
9998,"Dear user, this is an expires midnight notice ...",1


In [34]:
stacked_df.isna().sum()

text     1
label    0
dtype: int64

In [42]:
stacked_df.to_csv('../data/raw/phising_cleaned.csv', index = False)

In [11]:
stacked_raw = pd.concat([df_1,df_2], axis = 0)
stacked_raw.to_csv('../data/raw/stacked_raw.csv', index = False)

In [18]:
stacked_raw.isna().sum()

text     1
label    0
dtype: int64

In [ ]:
# Returns True if any value in the 'text' column is NaN
has_nan = df['text'].isna().any()

# Returns only the rows where 'text' is NaN
nan_rows = df[df['text'].isna()]



,text,label
138126,NaN,1


In [37]:
stacked_raw = stacked_raw.dropna()

In [39]:
stacked_raw.isna().sum()
stacked_raw.to_csv('../data/raw/stacked_raw.csv', index = False)

In [36]:
data_path = r'..\data\processed\email_dataset_processed.csv'

df = pd.read_csv(data_path)

df_cleaned = df.dropna()

print(df_cleaned.isna().sum())

text            0
label           0
cleaned_text    0
dtype: int64


In [29]:
df_cleaned.to_csv('../data/processed/email_dataset_processed.csv', index = False)